# CORIL - Difficulty Assignment (18 Indic translation pairs)

Assigns `difficulty` (Easy / Medium / Hard) to **200 rows sampled from each
input file**, so every language pair contributes to the result.

| | |
|---|---|
| Pairs | 19 - the 18 CORIL files (`dg-hi`, `en-hi`, `en-te`, `hi-dg`, `hi-en`, `hi-gu`, `hi-kn`, `hi-or`, `hi-pa`, `hi-sd`, `hi-te`, `hi-ur`, `kn-hi`, `or-hi`, `pa-hi`, `te-en`, `te-hi`, `ur-hi`) plus **`en-mni`** from `english_manipuri.jsonl` |
| Rows scored | 200 per pair = **3800** |
| Metric | chrF++ |

**One notebook, not nineteen.** Every file is the same task with the same
schema and metric; only the language pair differs, and that is data, not code.
Nineteen copies would drift apart the moment one needed a fix. English-Manipuri
is not a CORIL file, but it is the same shape, so `DATA_GLOBS` in Cell 3 simply
lists it alongside them - nothing else in the pipeline changes.

*Note: the output is still named `coril_difficulty.jsonl`. Rename `OUTPUT_FILE`
in Cell 3 if you would rather it reflected the wider scope.*

**Models are loaded once, not once per pair.** The loop is model-outer,
pair-inner: each of the three models is loaded a single time and scores all
3600 rows before being unloaded. The naive arrangement would load a model 54
times.

**Per-pair floors, because the pairs do not behave alike.** Copying the source
without translating scores:

| Pair | Copy-the-source chrF++ |
|---|---|
| `hi-dg` | **33.2%** |
| `dg-hi` | **27.2%** |
| the other 16 pairs | 2 - 7% |

Dogri is written in Devanagari and is close to Hindi, so a model can score well
on those two by barely changing the input. English-Manipuri crosses scripts
(Latin to Bengali), so its copy floor is about 2% - the unrelated-translation
floor binds there instead. Cell 4 measures the floor for each
pair from all 500 of its rows, and Cell 10 reports accuracy per pair against
its own floor.

**Runtime is the real constraint: 3800 rows x 3 models = 11,400 generations**,
roughly **5-7 hours** on a T4 - well beyond one Colab session. Progress is saved
per model *and* per pair after every batch, so you can close the tab and re-run:
completed work is skipped. Alternatively set `PAIRS` in Cell 3 to a handful and
work through them over several sessions.

**Output.** `coril_difficulty.jsonl` - all 14 schema fields with `difficulty`
filled in, every scored pair combined, plus a per-pair audit file.

### Cell 1 - Install dependencies and authenticate

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; Mistral is not.
Accept both licences on huggingface.co, create a **read** token, then add it in
Colab via the **key icon** as a secret named `HF_TOKEN`.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sacrebleu

import sacrebleu
print("sacrebleu", sacrebleu.__version__)

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")

### Cell 2 - Mount Drive

Drive holds the weight cache (three models, ~16.5 GB) **and** — because this run
spans several sessions — it is where you should keep the progress files. Set
`PROG_ON_DRIVE = True` in Cell 3 so scored rows survive a disconnected runtime;
otherwise they vanish with the VM and the whole run restarts.

In [ ]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights and progress are NOT preserved between")
    print("sessions. For a multi-hour run that means starting over each time.")

### Cell 3 - Configuration

- `PAIRS` - which language pairs to score. `None` means all 18. Set a short
  list to work through the corpus over several sessions.
- `N_PER_PAIR` - rows sampled from each pair's ~500. At 200 every pair
  contributes meaningfully; lower it for a faster first pass.
- `PROG_ON_DRIVE` - keep progress files on Drive so a disconnect costs nothing.
  Strongly recommended for the full run.
- `THRESHOLD_MODE` - `"auto_median"` by default; `"floor_margin"` anchors to
  the highest per-pair floor, which matters whenever `hi-dg` or `dg-hi` are in
  scope.

In [ ]:
import gc
import re
import json
import glob
import random
import shutil
import statistics
from collections import Counter, defaultdict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# CORIL plus the English-Manipuri corpus: same task, same schema, same
# metric, so it scores through the identical pipeline as a 19th pair.
DATA_GLOBS = ["coril_*.jsonl", "english_manipuri.jsonl"]

# ---- scope ----
PAIRS      = None        # None = all 19, or e.g. ["hi-te", "en-mni"]
N_PER_PAIR = 200

# ---- where progress lives ----
PROG_ON_DRIVE = True     # survives a disconnected Colab runtime

# ---- outputs ----
OUTPUT_FILE = "coril_difficulty.jsonl"
AUDIT_FILE  = "coril_audit.jsonl"

# ---- sampling / batching ----
SEED          = 42
MIN_REF_WORDS = 3
BATCH_SIZE    = 25
MAX_NEW_TOKENS = 256

# ---- thresholding ----
THRESHOLD_MODE  = "auto_median"     # auto_median | floor_margin | fixed
FLOOR_MARGIN    = 0.15
FIXED_THRESHOLD = 0.50

# ---- schema ----
SET_EVAL_METRIC = None              # None keeps the file's own "chrF++"

LANG_NAMES = {
    "hi": "Hindi", "en": "English", "te": "Telugu", "kn": "Kannada",
    "gu": "Gujarati", "or": "Odia", "pa": "Punjabi", "ur": "Urdu",
    "sd": "Sindhi", "dg": "Dogri", "mni": "Manipuri",
}

# measured, not assumed: Dogri shares Devanagari and much vocabulary with Hindi
HIGH_COPY_PAIRS = {"hi-dg", "dg-hi"}

if PROG_ON_DRIVE and DRIVE_OK:
    PROG_DIR = os.path.join(DRIVE_MOUNT, "MyDrive", "coril_progress")
else:
    PROG_DIR = "coril_progress"
    if PROG_ON_DRIVE:
        print("PROG_ON_DRIVE requested but Drive is not mounted - progress is")
        print("local and will be lost when the runtime ends.")
os.makedirs(PROG_DIR, exist_ok=True)

# ---- the three judges ----
USE_INSTRUCT = True
REPOS = {
    True: {"mistral": "mistralai/Mistral-7B-Instruct-v0.3",
           "llama":   "meta-llama/Llama-3.1-8B-Instruct",
           "gemma":   "google/gemma-2-9b-it"},
    False: {"mistral": "mistralai/Mistral-7B-v0.3",
            "llama":   "meta-llama/Llama-3.1-8B",
            "gemma":   "google/gemma-2-9b"},
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

print("progress dir: {}".format(PROG_DIR))
print("Judges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

### Cell 4 - Load every pair, measure floors, sample

Each file is read, filtered, and sampled independently, so **every pair
contributes `N_PER_PAIR` rows** rather than the larger files dominating a
pooled draw.

Floors are computed per pair from all ~500 of its rows, not from the 200
sampled - a floor needs more data than a sample provides. `PAIR_FLOOR` takes the
higher of copy-the-source and unrelated-translation, and Cell 9 and Cell 10
compare against it.

Read the printed table before starting a multi-hour run: a pair whose copy
floor is high is one where a good score may mean nothing.

In [ ]:
import sacrebleu
_CHRF = sacrebleu.CHRF(word_order=2)          # word_order=2 makes this chrF++


def chrf_pp(hypothesis, reference):
    if not hypothesis or not str(hypothesis).strip():
        return 0.0
    return _CHRF.sentence_score(str(hypothesis), [str(reference)]).score / 100.0


def usable(row):
    q = str(row.get("question") or "").strip()
    a = str(row.get("answer") or "").strip()
    return bool(q) and len(a.split()) >= MIN_REF_WORDS and q != a


files = []
for pattern in DATA_GLOBS:
    files += glob.glob(pattern)
files = sorted(set(files))
assert files, "no input files matched {} in the working directory".format(DATA_GLOBS)

by_pair, all_rows = {}, []
for path in files:
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    all_rows += rows
    pair = rows[0]["language"]
    assert {r["language"] for r in rows} == {pair}, "mixed pairs in " + path
    by_pair[pair] = [r for r in rows if usable(r)]

if PAIRS:
    missing = [p for p in PAIRS if p not in by_pair]
    assert not missing, "unknown pairs: {}".format(missing)
    by_pair = {p: v for p, v in by_pair.items() if p in PAIRS}

print("loaded {} files | {} pairs in scope".format(len(files), len(by_pair)))

# ---- per-pair floors, from every usable row of that pair ----
rng = random.Random(SEED)
PAIR_COPY, PAIR_RAND, PAIR_FLOOR = {}, {}, {}
for pair, rows in by_pair.items():
    sub = rows if len(rows) <= 80 else rng.sample(rows, 80)
    PAIR_COPY[pair] = statistics.mean(chrf_pp(r["question"], r["answer"]) for r in sub)
    PAIR_RAND[pair] = statistics.mean(
        chrf_pp(sub[(i + 3) % len(sub)]["answer"], r["answer"]) for i, r in enumerate(sub))
    PAIR_FLOOR[pair] = max(PAIR_COPY[pair], PAIR_RAND[pair])

# ---- sample N_PER_PAIR from each pair ----
sample, per_pair_sample = [], {}
for pair in sorted(by_pair):
    rows = by_pair[pair]
    n = min(N_PER_PAIR, len(rows))
    random.seed(SEED)
    picked = random.sample(rows, n)
    per_pair_sample[pair] = picked
    sample += picked

MAX_FLOOR = max(PAIR_FLOOR[p] for p in per_pair_sample)

print("\n  {:<8} {:>5} {:>7} {:>10} {:>11} {:>9}".format(
    "pair", "avail", "taken", "copy-src", "unrelated", "floor"))
for pair in sorted(by_pair):
    warn = "  <- copying scores well" if pair in HIGH_COPY_PAIRS else ""
    print("  {:<8} {:>5} {:>7} {:>9.1%} {:>11.1%} {:>9.1%}{}".format(
        pair, len(by_pair[pair]), len(per_pair_sample[pair]),
        PAIR_COPY[pair], PAIR_RAND[pair], PAIR_FLOOR[pair], warn))

print("\ntotal rows to score: {} x {} models = {} generations".format(
    len(sample), len(MODELS), len(sample) * len(MODELS)))
print("highest per-pair floor in scope: {:.1%}".format(MAX_FLOOR))
assert len({r["id"] for r in sample}) == len(sample), "duplicate ids across pairs"

### Cell 5 - Build the translation prompt

The prompt names both languages, resolved from the row's own pair code -
"Translate Hindi into Telugu", never a generic "translate". With 18 pairs a
generic instruction reliably produces the wrong target language.

**Few-shot examples come from the same pair**, drawn from rows outside that
pair's sample. A Telugu example teaches nothing about Odia and would show the
wrong target script.

In [ ]:
def flat(text):
    return " ".join(str(text).split())


def names_of(pair):
    s, t = pair.split("-")
    return LANG_NAMES.get(s, s), LANG_NAMES.get(t, t)


def pick_fewshot(pair, k=3):
    used = {r["id"] for r in per_pair_sample.get(pair, [])}
    cand = [r for r in by_pair[pair]
            if r["id"] not in used and 5 <= len(str(r["question"]).split()) <= 30]
    random.Random(SEED + 1).shuffle(cand)
    return cand[:k]


FEWSHOT_BY_PAIR = {p: pick_fewshot(p) for p in sorted(per_pair_sample)}
print("few-shot per pair (all from OUTSIDE that pair's sample):")
print("  {}".format({p: len(v) for p, v in sorted(FEWSHOT_BY_PAIR.items())}))


def instructions(pair):
    s, t = names_of(pair)
    return ("Translate {} into {}.\n\n"
            "Write natural, fluent {} in its own script. Keep proper names and "
            "technical terms in their usual {} forms.\n\n"
            "Output only the {} translation - no transliteration, no "
            "explanation, no repetition of the {}.").format(s, t, t, t, t, s)


def build_completion(row):
    pair = row["language"]
    s, t = names_of(pair)
    text = instructions(pair) + "\n"
    for ex in FEWSHOT_BY_PAIR.get(pair, []):
        text += "\n{}: {}\n{}: {}\n".format(s, flat(ex["question"]), t, flat(ex["answer"]))
    text += "\n{}: {}\n{}:".format(s, flat(row["question"]), t)
    return text


def build_chat_messages(row):
    pair = row["language"]
    msgs = [{"role": "system", "content": instructions(pair)}]
    for ex in FEWSHOT_BY_PAIR.get(pair, []):
        msgs.append({"role": "user", "content": flat(ex["question"])})
        msgs.append({"role": "assistant", "content": flat(ex["answer"])})
    msgs.append({"role": "user", "content": flat(row["question"])})
    return msgs


print("\n" + "=" * 70)
print(build_completion(sample[0]))
print("=" * 70)
print("[reference: {}]".format(flat(sample[0]["answer"])))

### Cell 6 - Generate and score

Greedy decoding for reproducibility. `clean_translation` strips a leading label
and cuts anything from a following language marker, which base models emit as
they continue the few-shot pattern. It does not repair wrong-language output -
`looks_like_source` records that separately, so Cell 10 can tell a model that
echoed the input from one that translated badly.

In [ ]:
_NAMES_RE = "|".join(sorted(set(LANG_NAMES.values())))
_GENERIC = re.compile(r"^\s*(here(?:\s+is|'s)?\s+the\s+)?(translation|output)\s*[:\-]\s*", re.I)
_LANGLAB = re.compile(r"^\s*(" + _NAMES_RE + r")\s*[:\-]\s*", re.I)
_MARKER  = re.compile(r"\n\s*(" + _NAMES_RE + r")\s*:", re.I)


def clean_translation(text):
    t = (text or "").strip()
    t = _MARKER.split(t)[0]
    for line in t.split("\n"):
        line = line.strip()
        if not line:
            continue
        line = _GENERIC.sub("", line)
        line = _LANGLAB.sub("", line)
        line = line.strip().strip('"\u2018\u2019\u201c\u201d')
        if line:
            return " ".join(line.split())
    return ""


def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def looks_like_source(hyp, row):
    return chrf_pp(hyp, row["question"]) > 0.80


@torch.no_grad()
def translate(model, tokenizer, row):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(row)
    else:
        msgs = build_chat_messages(row)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    budget = min(4 * len(str(row["question"]).split()) + 48, MAX_NEW_TOKENS)
    out = model.generate(**inputs, max_new_tokens=budget, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return clean_translation(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()


for raw, want in [("Telugu: hello", "hello"),
                  ("Translation: hi there", "hi there"),
                  ("good\nEnglish: next", "good"),
                  ('"quoted"', "quoted"), ("", "")]:
    got = clean_translation(raw)
    assert got == want, (raw, got, want)
print("Generation and scoring functions defined")

### Cell 7 - Load-or-cache, and the model-outer runner

`run_all_for_model` is where the multi-session design lives. For one model it
loads the weights **once**, then walks every pair in scope, appending each batch
to `coril_progress/<model>__<pair>.jsonl` before starting the next.

Two consequences worth knowing:

- Each model is loaded once rather than once per pair - three loads instead of
  fifty-four.
- Because progress is keyed by model *and* pair, an interrupted run resumes at
  batch granularity. Re-running after a disconnect skips everything already
  scored and picks up mid-pair.

With `PROG_ON_DRIVE = True` those files sit on Drive, so a dead runtime costs
nothing.

In [ ]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def prog_path(model_name, pair):
    return os.path.join(PROG_DIR, "{}__{}.jsonl".format(model_name, pair))


def load_progress(model_name, pair):
    p = prog_path(model_name, pair)
    done = {}
    if os.path.exists(p):
        with open(p, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
    return done


def run_all_for_model(spec):
    """Load one model, score every pair in scope, then unload."""
    todo = {}
    for pair, rows in sorted(per_pair_sample.items()):
        done = load_progress(spec["name"], pair)
        remaining = [r for r in rows if r["id"] not in done]
        if remaining:
            todo[pair] = remaining
    if not todo:
        print("  {} already complete for every pair - skipping load".format(spec["name"]))
        return

    outstanding = sum(len(v) for v in todo.values())
    print("  {} rows outstanding across {} pairs".format(outstanding, len(todo)))
    model, tokenizer = load_model(spec)

    for pair in sorted(todo):
        rows = todo[pair]
        nb = (len(rows) + BATCH_SIZE - 1) // BATCH_SIZE
        for start in range(0, len(rows), BATCH_SIZE):
            batch, results = rows[start:start + BATCH_SIZE], []
            for row in batch:
                hyp = translate(model, tokenizer, row)
                results.append({"id": row["id"], "pair": pair,
                                "chrf": chrf_pp(hyp, row["answer"]),
                                "n_words": len(hyp.split()),
                                "echoed": int(looks_like_source(hyp, row)),
                                "output": hyp})
            with open(prog_path(spec["name"], pair), "a", encoding="utf-8") as f:
                for item in results:
                    f.write(json.dumps(item, ensure_ascii=False) + "\n")
            m = sum(x["chrf"] for x in results) / len(results)
            print("    {:<8} batch {}/{} saved | batch mean chrF++ {:.1%}".format(
                pair, start // BATCH_SIZE + 1, nb, m))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))

print("Runner defined")

### Cell 8 - Run all three models

The long cell - roughly **5-7 hours** for the full 18 pairs. Re-run it as many
times as you need: anything already scored is skipped, so it resumes rather
than restarting.

If a session is likely to time out, run a subset by setting `PAIRS` in Cell 3
and re-running that cell first.

In [ ]:
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    run_all_for_model(spec)

# gather everything scored so far
preds = {s["name"]: {} for s in MODELS}
for s in MODELS:
    for pair in per_pair_sample:
        preds[s["name"]].update(load_progress(s["name"], pair))

complete = [r for r in sample
            if all(r["id"] in preds[s["name"]] for s in MODELS)]
print("\nrows scored by all three models: {}/{}".format(len(complete), len(sample)))
if len(complete) < len(sample):
    print("  re-run this cell to finish the rest - progress is preserved")

### Cell 9 - Find the threshold

One threshold across all pairs, applied only to rows every model has scored.

`"auto_median"` takes the pooled median. `"floor_margin"` anchors to the highest
per-pair floor in scope plus `FLOOR_MARGIN` - use it whenever `hi-dg` or
`dg-hi` are included, since a threshold below their copy floor would pass a
model that merely echoed the Hindi.

In [ ]:
assert complete, "nothing scored yet - run Cell 8"
pooled = sorted(preds[s["name"]][r["id"]]["chrf"] for r in complete for s in MODELS)

def pct(p):
    return pooled[min(len(pooled) - 1, int(p * len(pooled)))]

print("chrF++ per model (over {} fully-scored rows):".format(len(complete)))
print("  {:<10} {:>7} {:>7} {:>7} {:>7}".format("model", "p25", "median", "p75", "mean"))
for s in MODELS:
    v = sorted(preds[s["name"]][r["id"]]["chrf"] for r in complete)
    print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
        s["name"], v[len(v) // 4], v[len(v) // 2], v[3 * len(v) // 4], sum(v) / len(v)))
print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
    "POOLED", pct(.25), pct(.50), pct(.75), sum(pooled) / len(pooled)))
print("\nhighest per-pair floor in scope: {:.1%}".format(MAX_FLOOR))


def difficulty_at(th):
    out = Counter()
    for r in complete:
        votes = sum(preds[s["name"]][r["id"]]["chrf"] >= th for s in MODELS)
        out["Easy" if votes == 3 else ("Medium" if votes == 2 else "Hard")] += 1
    return out


if THRESHOLD_MODE == "auto_median":
    THRESHOLD, why = pct(.50), "median of all pooled model scores"
elif THRESHOLD_MODE == "floor_margin":
    THRESHOLD = MAX_FLOOR + FLOOR_MARGIN
    why = "highest per-pair floor + {:.2f}".format(FLOOR_MARGIN)
elif THRESHOLD_MODE == "fixed":
    THRESHOLD, why = FIXED_THRESHOLD, "FIXED_THRESHOLD from Cell 3"
else:
    raise ValueError("unknown THRESHOLD_MODE: " + str(THRESHOLD_MODE))

print("\nsensitivity:")
print("  {:>9}  {:>6} {:>7} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
for th in sorted(set(round(x, 3) for x in
                     [.20, .30, .40, .50, .60, .70,
                      round(MAX_FLOOR, 3), round(THRESHOLD, 3)])):
    d = difficulty_at(th)
    tag = "  <- CHOSEN" if abs(th - round(THRESHOLD, 3)) < 1e-9 else ""
    if abs(th - round(MAX_FLOOR, 3)) < 1e-9:
        tag += "  (highest pair floor)"
    print("  {:>9.3f}  {:>6} {:>7} {:>6}{}".format(
        th, d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0), tag))

print("\nTHRESHOLD = {:.3f}  ({})".format(THRESHOLD, why))
if THRESHOLD <= MAX_FLOOR:
    print("  WARNING: below the highest per-pair floor ({:.1%}) - on that pair a".format(MAX_FLOOR))
    print("  model could pass without translating. Use THRESHOLD_MODE='floor_margin'.")
else:
    print("  sits {:.1f} points above the highest per-pair floor - OK".format(
        100 * (THRESHOLD - MAX_FLOOR)))
_b = difficulty_at(THRESHOLD)
if min(_b.get(k, 0) for k in ("Easy", "Medium", "Hard")) == 0:
    print("  WARNING: one difficulty band is empty - pick another value.")

### Cell 10 - Write the schema, verify, and report per pair

Votes sum into Easy / Medium / Hard. Output rows are rebuilt key-by-key from
`SCHEMA_KEYS`, so all 14 fields survive in schema order across every pair.

The per-pair table is the one to read. A pair whose mean score barely clears its
own floor is not being translated, however healthy the overall average looks -
and `hi-dg` / `dg-hi` will flatter a model that only copies.

In [ ]:
def get_difficulty(votes):
    score = sum(votes)
    return "Easy" if score == 3 else ("Medium" if score == 2 else "Hard")


final_results, audit = [], []
for row in complete:
    scores = [preds[s["name"]][row["id"]]["chrf"] for s in MODELS]
    votes  = [int(x >= THRESHOLD) for x in scores]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id": row["id"], "pair": row["language"],
        "pair_floor": round(PAIR_FLOOR[row["language"]], 4),
        "difficulty": difficulty, "votes": votes,
        "chrf": [round(x, 4) for x in scores],
        "threshold": round(THRESHOLD, 4),
        "source": " ".join(str(row["question"]).split()),
        "reference": " ".join(str(row["answer"]).split()),
        "translations": {s["name"]: preds[s["name"]][row["id"]]["output"] for s in MODELS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))

src_by_id = {r["id"]: r for r in all_rows}
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
altered  = [r["id"] for r in final_results
            if r["question"] != src_by_id[r["id"]]["question"]
            or r["answer"] != src_by_id[r["id"]]["answer"]]
print("\nSchema check : {} rows | wrong keys: {} | null difficulty: {} | altered: {}".format(
    len(final_results), len(bad_keys),
    sum(1 for r in final_results if r["difficulty"] is None), len(altered)))

dist, total = Counter(r["difficulty"] for r in final_results), len(final_results)
print("\nDifficulty distribution (threshold {:.3f}):".format(THRESHOLD))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:5d}  ({:.1f}%)".format(level, n, n / total * 100))

print("\nPer-model:")
for s in MODELS:
    v = [preds[s["name"]][r["id"]] for r in complete]
    m = sum(x["chrf"] for x in v) / len(v)
    echo = sum(x["echoed"] for x in v)
    empty = sum(1 for x in v if x["n_words"] == 0)
    flags = []
    if echo / total > 0.10:
        flags.append("echoing the source")
    if empty / total > 0.05:
        flags.append("empty outputs")
    print("  {:<10} mean chrF++ {:.1%} | echoed {:>4}/{} | empty {:>4}/{}{}".format(
        s["name"], m, echo, total, empty, total,
        "  <- " + ", ".join(flags) if flags else ""))

print("\nPer-pair mean chrF++ vs that pair's own floor:")
print("  {:<8} {:>5} {:>8}".format("pair", "n", "floor") +
      "".join("{:>10}".format(s["name"][:8]) for s in MODELS) + "   E/M/H")
by_p = defaultdict(list)
for r in final_results:
    by_p[r["language"]].append(r["id"])
for pair in sorted(by_p):
    ids = by_p[pair]
    line = "  {:<8} {:>5} {:>7.1%}".format(pair, len(ids), PAIR_FLOOR[pair])
    weak = False
    for s in MODELS:
        m = sum(preds[s["name"]][i]["chrf"] for i in ids) / len(ids)
        line += "{:>10.1%}".format(m)
        if m <= PAIR_FLOOR[pair]:
            weak = True
    d = Counter(r["difficulty"] for r in final_results if r["language"] == pair)
    line += "   {}/{}/{}".format(d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0))
    if weak:
        line += "  <- at/below floor"
    print(line)